# Московский старт — интерактивная текстовая игра

Запустите все ячейки в ноутбуке и вводите ответы в соответствии с подсказками (номера пунктов или `да`/`нет`).


In [ ]:
TITLE = "МОСКОВСКИЙ СТАРТ"
START_MONEY = 50000
HOSTEL_COST = 5000

ENDINGS = {
    "specialist": "Специалист — вы закончили учёбу, стали разработчиком и строите карьеру.",
    "businessman": "Бизнесмен — ваш проект в Москве растёт и приносит прибыль.",
    "family": "Семья — вы нашли счастье в столице и живёте в балансе между работой и домом.",
    "prison": "Тюрьма (15 лет) — рискованные схемы оборачиваются сроком.",
    "home": "Возвращение домой — деньги закончились, и вы уехали обратно.",
    "debt": "Долговое рабство — кредиты и проценты полностью вас поглотили."
}

BRANCH_NAMES = {
    "start": "Старт",
    "work": "Работа",
    "business": "Бизнес",
    "study": "Учёба"
}


def ask_choice(prompt, options):
    while True:
        print(prompt)
        for num, text in options.items():
            print(f"{num}) {text}")
        answer = input("Ваш выбор: ").strip()
        if answer.isdigit():
            number = int(answer)
            if number in options:
                return number
        print("Пожалуйста, введите номер из списка.\n")


def ask_yes_no(prompt):
    while True:
        answer = input(f"{prompt} (да/нет): ").strip().lower()
        if answer in ("да", "нет"):
            return answer
        print("Нужно ответить 'да' или 'нет'.\n")


def advance_time(stats, months, reason):
    stats["months"] += months
    stats["time_log"].append((reason, months))


def add_income(stats, amount, reason, branch):
    stats["cash"] += amount
    stats["income_total"] += amount
    stats["income_log"].append((reason, amount, branch))
    stats["branch_totals"][branch]["income"] += amount


def add_expense(stats, amount, reason, branch):
    stats["cash"] -= amount
    stats["expense_total"] += amount
    stats["expense_log"].append((reason, amount, branch))
    stats["branch_totals"][branch]["expense"] += amount


def add_credit(stats, amount, debt_amount, reason, branch):
    stats["cash"] += amount
    stats["debt"] += debt_amount
    stats["credit_log"].append((reason, amount, debt_amount, branch))
    stats["debt_log"].append((reason, debt_amount))
    stats["branch_totals"][branch]["income"] += amount


def increase_debt(stats, amount, reason):
    stats["debt"] += amount
    stats["debt_log"].append((reason, amount))


def format_months(months):
    years = months // 12
    remaining = months % 12
    parts = []
    if years:
        parts.append(f"{years} г.")
    if remaining or not parts:
        parts.append(f"{remaining} мес.")
    return " ".join(parts)


def print_finance_report(stats):
    print("\nФИНАНСОВЫЙ ОТЧЁТ")
    print(f"Деньги на руках: {stats['cash']}₽")
    print(f"Доходы всего: +{stats['income_total']}₽")
    print(f"Траты всего: -{stats['expense_total']}₽")
    print(f"Текущий долг по кредитам: {stats['debt']}₽")
    print(f"Время в Москве: {format_months(stats['months'])}")

    print("\nСуммы денег по веткам (доход / траты / итог):")
    for branch_key, branch_name in BRANCH_NAMES.items():
        income = stats["branch_totals"][branch_key]["income"]
        expense = stats["branch_totals"][branch_key]["expense"]
        net = income - expense
        print(f"- {branch_name}: +{income}₽ / -{expense}₽ / {net}₽")

    print("\nВсе траты:")
    if stats["expense_log"]:
        for idx, (reason, amount, branch) in enumerate(stats["expense_log"], 1):
            print(f"{idx}. {reason} ({BRANCH_NAMES[branch]}): -{amount}₽")
    else:
        print("Трат нет.")

    print("\nКредиты и рост долга:")
    if stats["debt_log"]:
        for idx, (reason, amount) in enumerate(stats["debt_log"], 1):
            print(f"{idx}. {reason}: +{amount}₽ к долгу")
    else:
        print("Кредитов нет.")

    print("\nХод времени:")
    if stats["time_log"]:
        for idx, (reason, months) in enumerate(stats["time_log"], 1):
            print(f"{idx}. +{months} мес. — {reason}")
    else:
        print("Сюжет развивался в рамках стартового дня.")


def save_result(log, ending_key, ending_text, events, stats):
    with open("game_result.txt", "w", encoding="utf-8") as file:
        file.write(f"{TITLE} — ИТОГ\n")
        file.write(f"Остаток денег: {stats['cash']}₽\n")
        file.write(f"Доходы: +{stats['income_total']}₽\n")
        file.write(f"Траты: -{stats['expense_total']}₽\n")
        file.write(f"Долг по кредитам: {stats['debt']}₽\n")
        file.write(f"Время в Москве: {format_months(stats['months'])}\n")

        file.write("\nСуммы денег по веткам (доход / траты / итог):\n")
        for branch_key, branch_name in BRANCH_NAMES.items():
            income = stats["branch_totals"][branch_key]["income"]
            expense = stats["branch_totals"][branch_key]["expense"]
            net = income - expense
            file.write(f"- {branch_name}: +{income}₽ / -{expense}₽ / {net}₽\n")

        file.write("\nВсе траты:\n")
        if stats["expense_log"]:
            for idx, (reason, amount, branch) in enumerate(stats["expense_log"], 1):
                file.write(f"{idx}. {reason} ({BRANCH_NAMES[branch]}): -{amount}₽\n")
        else:
            file.write("Трат нет.\n")

        file.write("\nКредиты и рост долга:\n")
        if stats["debt_log"]:
            for idx, (reason, amount) in enumerate(stats["debt_log"], 1):
                file.write(f"{idx}. {reason}: +{amount}₽ к долгу\n")
        else:
            file.write("Кредитов нет.\n")

        file.write("\nХод времени:\n")
        if stats["time_log"]:
            for idx, (reason, months) in enumerate(stats["time_log"], 1):
                file.write(f"{idx}. +{months} мес. — {reason}\n")
        else:
            file.write("Сюжет развивался в рамках стартового дня.\n")

        file.write("\nВыборы игрока:\n")
        for idx, entry in enumerate(log, 1):
            file.write(f"Выбор {idx}: {entry}\n")

        file.write("\nСобытия веток: " + ", ".join(sorted(events)) + "\n")
        file.write(f"\nКОНЦОВКА [{ending_key}]: {ending_text}\n")

    print("\n---")
    print("Результат сохранён в файл game_result.txt")
    print(ending_text)


def handle_work(log, events, stats):
    events.add("work")
    log.append("Работа: курьер в сервисе доставки")

    advance_time(stats, 1, "Первая неделя работы")
    add_income(stats, 14000, "Заработок за первую неделю курьером", "work")
    print("\nПервая неделя: вы заработали 14000₽ на доставке.")

    risky_offer = ask_yes_no("Коллега предлагает серую схему заработка. Вписываетесь?")
    if risky_offer == "да":
        log.append("Согласился на серую схему")
        events.add("risk")

        advance_time(stats, 1, "Месяц серой схемы")
        add_income(stats, 66000, "Доход от серой схемы за месяц", "work")
        advance_time(stats, 1, "Расследование")
        ending_key = "prison"
    else:
        log.append("Отказался от схемы и растёшь по карьере")
        events.add("career")

        advance_time(stats, 12, "Год стабильной работы")
        add_income(stats, 720000, "Зарплата и премии за год", "work")
        ending_key = "family"

    return ending_key


def handle_business(log, events, stats):
    events.add("business")
    log.append("Бизнес: старт в Москве")

    options = {
        1: "Открыть интернет-магазин",
        2: "Взять кредит и открыть кофейню"
    }
    biz_choice = ask_choice("Какой бизнес выберете?", options)

    if biz_choice == 1:
        log.append("Запустил интернет-магазин")
        events.add("online_shop")

        add_expense(stats, 25000, "Стартовые вложения в интернет-магазин", "business")
        advance_time(stats, 1, "Первый месяц интернет-магазина")
        add_expense(stats, 10000, "Убыток первого месяца интернет-магазина", "business")

        advance_time(stats, 1, "Второй месяц интернет-магазина")
        add_income(stats, 5000, "Прибыль второго месяца интернет-магазина", "business")

        supplier_deal = ask_yes_no("Поставщик предлагает товар без документов. Берём?")
        if supplier_deal == "да":
            log.append("Взял товар без документов")
            events.add("tax_trouble")

            advance_time(stats, 6, "Продажи без документов")
            add_income(stats, 500000, "Прибыль за 6 месяцев на серых поставках", "business")
            add_expense(stats, 1000000, "Штраф налоговой", "business")
            ending_key = "prison"
        else:
            log.append("Работаешь легально и масштабируешься")
            events.add("scale")

            advance_time(stats, 12, "Год легального роста")
            add_income(stats, 960000, "Прибыль за год (80000₽ в месяц)", "business")
            ending_key = "businessman"
    else:
        log.append("Открыл кофейню на кредитные деньги")
        events.add("coffee_shop")

        add_credit(stats, 150000, 150000, "Кредит на запуск кофейни", "business")
        add_expense(stats, 150000, "Открытие кофейни", "business")

        advance_time(stats, 6, "Полгода попыток развить кофейню")
        increase_debt(stats, 150000, "Проценты по кредиту кофейни")
        add_expense(stats, 60000, "Операционные убытки кофейни", "business")

        log.append("Бизнес не идёт, траты растут")
        ending_key = "home"

    return ending_key


def handle_study(log, events, stats):
    events.add("study")
    log.append("Учёба: поступил на IT-специальность")

    add_expense(stats, 40000, "Оплата семестра в университете", "study")
    advance_time(stats, 1, "Первый месяц учёбы")

    study_options = {
        1: "Взять академический отпуск и пойти работать",
        2: "Взять кредит на учёбу",
        3: "Подработка по специальности"
    }
    study_choice = ask_choice("Как удержаться на плаву?", study_options)

    if study_choice == 1:
        log.append("Берёте академ и ищете работу")
        events.add("study_to_work")

        advance_time(stats, 1, "Академический отпуск")
        return handle_work(log, events, stats)

    if study_choice == 2:
        log.append("Берёте кредит, долг растёт")
        events.add("loan")

        add_credit(stats, 300000, 300000, "Образовательный кредит", "study")
        add_expense(stats, 300000, "Оплата учёбы из кредитных денег", "study")

        advance_time(stats, 48, "4 года учёбы в университете")
        increase_debt(stats, 200000, "Проценты по образовательному кредиту")
        return "debt"

    log.append("Находите подработку по специальности")
    events.add("part_time")

    advance_time(stats, 12, "Учёба и подработка по специальности")
    add_income(stats, 360000, "Подработка по специальности (30000₽ в месяц)", "study")

    part_time_options = {
        1: "Стажёр в IT-компании",
        2: "Бросить учёбу и открыть бизнес"
    }
    sub_choice = ask_choice("Какую подработку выберете?", part_time_options)

    if sub_choice == 1:
        log.append("Идёте стажёром и растёте до специалиста")
        events.add("intern")

        advance_time(stats, 3, "Стажировка аналитики данных")
        add_income(stats, 120000, "Стажировка (40000₽ x 3 месяца)", "study")
        return "specialist"

    log.append("Бросаете учёбу и стартуете бизнес")
    events.add("study_to_business")
    return handle_business(log, events, stats)


In [ ]:
log = []
events = set()
events.update(["start", "hostel"])

stats = {
    "cash": START_MONEY,
    "debt": 0,
    "months": 0,
    "income_total": 0,
    "expense_total": 0,
    "income_log": [],
    "expense_log": [],
    "credit_log": [],
    "debt_log": [],
    "time_log": [],
    "branch_totals": {
        key: {"income": 0, "expense": 0}
        for key in BRANCH_NAMES
    }
}

print(f"{TITLE}: у вас {START_MONEY}₽. Вы платите за хостел {HOSTEL_COST}₽.")
add_expense(stats, HOSTEL_COST, "Оплата хостела", "start")
log.append(f"Оплатил хостел {HOSTEL_COST}₽, осталось {stats['cash']}₽")

main_options = {
    1: "Устроиться на работу",
    2: "Поступить в университет",
    3: "Начать бизнес"
}
first_choice = ask_choice("С чего начнём жизнь в столице?", main_options)
log.append(main_options[first_choice])

if first_choice == 1:
    ending_key = handle_work(log, events, stats)
elif first_choice == 2:
    ending_key = handle_study(log, events, stats)
else:
    ending_key = handle_business(log, events, stats)

ending_text = ENDINGS[ending_key]
log_copy = log.copy()
choices_dict = {idx + 1: text for idx, text in enumerate(log_copy)}

print("\nВаш путь:")
for number, text in choices_dict.items():
    print(f"{number}. {text}")

print_finance_report(stats)
save_result(log_copy, ending_key, ending_text, events, stats)
